# 🧱 Lesson 03: Parsing

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HosseinHeydari2004/RAG-Tutorial-From-Zero-to-Advanced/blob/main/notebooks/03_parsing.ipynb)

> **Loading gave us text. Parsing gives us *structure*: which part is a title, and which part belongs to it.**

---

## 🎯 Goal
By the end of this lesson you can split a PDF into **sections** (one per heading) and keep each section's title as metadata.

## ℹ️ Lesson info

| | |
|---|---|
| ⏱ **Time** | ~20 minutes |
| 📚 **Prerequisites** | [Lesson 02](02_loading.ipynb) |
| 🔑 **API key** | Not needed |
| 🛠 **Libraries** | `pdfplumber`, `langchain-core` |

In [ ]:
!pip install -q -U pdfplumber langchain-core

In [1]:
from pathlib import Path
import urllib.request

PDF_URL = "https://raw.githubusercontent.com/HosseinHeydari2004/RAG-Tutorial-From-Zero-to-Advanced/main/data/pdf1.pdf"

pdf_path = next((p for p in [Path("data/pdf1.pdf"), Path("../data/pdf1.pdf")] if p.exists()), None)

if pdf_path is None:
    pdf_path = Path("data/pdf1.pdf")
    pdf_path.parent.mkdir(exist_ok=True)
    urllib.request.urlretrieve(PDF_URL, pdf_path)

print("using:", pdf_path)

using: ..\data\pdf1.pdf


## 💡 The idea: a heading is just bigger text

A PDF doesn't know what a "heading" is. It only knows *"draw this text at this size, here"*. But humans make headings **bigger** than body text, and we can use that.

Plan:
1. Read every line together with its **font size**.
2. The most common size is the **body text**.
3. A line **bigger than the body** is a heading, and it starts a new section.

First step: read the lines and their sizes.

In [3]:
import pdfplumber
from collections import Counter

lines = []
with pdfplumber.open(pdf_path) as pdf:
    for page_number, page in enumerate(pdf.pages):
        for line in page.extract_text_lines():
            size = round(max(char["size"] for char in line["chars"]), 1)
            lines.append({"text": line["text"], "size": size, "page": page_number})

print("size counts:", Counter(line["size"] for line in lines))
print()
for line in lines[:4]:
    print(line["size"], "|", line["text"][:60])

size counts: Counter({10.0: 20, 14.0: 4, 18.0: 1})

18.0 | Employee Handbook
14.0 | Non-Disclosure Agreement (NDA) Policy
10.0 | Employees must protect confidential information belonging to
10.0 | This includes, but is not limited to, product roadmaps, cust


## ✂️ Find the headings and build sections

Now we apply the plan:

1. The most common size is the **body size** (here 10).
2. Any line **bigger than the body** is a heading and starts a new section.
3. Lines after a heading belong to that heading.

The big "Employee Handbook" title has no text under it before the next heading, so we drop empty sections.

In [4]:
body_size = Counter(line["size"] for line in lines).most_common(1)[0][0]
print("body size:", body_size)

sections = []
current = None
for line in lines:
    if line["size"] > body_size:
        current = {"title": line["text"], "page": line["page"], "text": []}
        sections.append(current)
    elif current is not None:
        current["text"].append(line["text"])

sections = [section for section in sections if section["text"]]

for section in sections:
    print(section["title"], "|", len(" ".join(section["text"])), "characters")

body size: 10.0
Non-Disclosure Agreement (NDA) Policy | 427 characters
Workplace Conduct Policy | 360 characters
Paid Time Off (PTO) Policy | 582 characters
Travel & Expense Policy | 462 characters


## 📦 Sections → Documents

Same shape as in Lesson 02, but now each `Document` is one **section**, and its title is saved in the metadata. Later, this lets us show "from section *Paid Time Off (PTO) Policy*" next to an answer.

In [5]:
from langchain_core.documents import Document

parsed_documents = [
    Document(
        page_content=" ".join(section["text"]),
        metadata={
            "source": pdf_path.name,
            "page": section["page"],
            "section": section["title"],
        },
    )
    for section in sections
]

for doc in parsed_documents:
    print(doc.metadata["section"])
    print("  ", doc.page_content[:90], "...")

Non-Disclosure Agreement (NDA) Policy
   Employees must protect confidential information belonging to the company, its clients, and ...
Workplace Conduct Policy
   Employees must maintain a respectful, professional environment free from harassment, discr ...
Paid Time Off (PTO) Policy
   Fullntime employees accrue PTO according to the following schedule: (cid:127) 0–1 years of ...
Travel & Expense Policy
   Employees may be reimbursed for reasonable and necessary expenses incurred during approved ...


### 👀 What we saw

We now have 4 clean **sections**, each with its title in the metadata. But look closely at the text of *Paid Time Off (PTO) Policy*:

| Defect | What we got | What it should be |
|---|---|---|
| Hyphen in "Full-time" | `Fullntime` here, `Full■time` with `pypdf` in Lesson 02 | `Full-time` |
| Bullet points | `(cid:127)` | `•` |

Two lessons here:
1. **Different parsers fail in different ways** on the *same* file. There is no perfect library.
2. Parsing gives structure, **not clean text**. Cleaning the text is the next lesson.

> `(cid:N)` usually means the library found a character in the PDF but the font gave it no usable Unicode mapping.

## ⚠️ What this method can't do

The font-size trick is simple and transparent, which is why we use it to learn. But:

- **Same-size headings:** if a heading is *bold* but the same size as the body, we miss it.
- **Scanned PDFs:** a scan is an image, so there is no text and no font size. You need OCR.
- **Tables and multiple columns:** lines from different columns can get mixed together.

Real projects often use more complete tools for this (for example `unstructured`, or parsers that work with page layout). We don't use them here, but know they exist.

## 📌 Summary

- **Parsing** goes beyond reading text: it recovers *structure* such as titles and sections.
- A PDF has no real "heading" concept. We guessed headings from **font size**: bigger than the body text means a heading.
- Each section became a `Document` with `section` in its metadata, ready for filtering and citations later.
- Parsers are imperfect and fail differently (`Full■time` vs `Fullntime`, `(cid:127)` for bullets). **Always inspect the result.**

## ✍️ Exercise (10 minutes)

In the cell that builds `parsed_documents`, we joined lines with `" "`. That glued the bullet points into one paragraph.

1. Change it to `"\n".join(...)` and print the *Paid Time Off (PTO) Policy* document.
2. Compare the two versions. Which one keeps the list readable?

**Success:** you can explain in one sentence what we gain and what we lose with each choice.

💡 Solution: `solutions/03_parsing.ipynb` (try on your own first)

## 🔥 Challenge (optional)

Our rule says "bigger than body = heading". Find a PDF where this fails (for example, one with bold headings of the same size as the body text). Describe *what* goes wrong, then think of one other signal you could use to detect headings (bold font? all caps? line position?).

## 📖 Next

We have sections, but the text inside still contains defects. In the next lesson we **clean** it.

**➡️ Next lesson:** [04 Cleaning](04_cleaning.ipynb)